# Week 2 Lab — Vector Geometry, CRS, and Spatial Decisions

**Course:** Geo-Data and Geo-Data Management  
**Lab focus:** use Shapely and GeoPandas to represent geographic features, measure them in an appropriate CRS, and support a simple spatial decision.


## What you will do today

Imagine you are helping a university research team plan an **urban microclimate field survey** near Vienna. The team will walk from an agreed meeting point to a survey stop, then collect observations within a predefined survey zone. **We are learning to represent a plan, not discovering actual streets or measuring real field sites.**

We start with **one pilot visit** (guided practice). Afterwards, you will compare **three new candidate survey zones** and recommend one (independent checkpoint).

By the end, you should be able to:

1. create and inspect **Point**, **LineString**, and **Polygon** objects;
2. connect geometries and attributes in a **GeoDataFrame**;
3. explain **CRS assignment vs coordinate transformation**;
4. calculate distance, provided route length, and area in meaningful units;
5. visualize the data and justify a simple **field-survey decision**.

All coordinates and features below are **synthetic Vienna-like teaching examples**, not verified roads, survey plots or pedestrian routes.

## Before you start

Make sure the notebook is using the **IT_Geo** kernel. Run the cells from top to bottom.

The guided examples give you working patterns. The checkpoint uses **different input data**: you will adapt what you learned to make and justify one geographic decision. No external data download is required.


## Guided Lab — the pilot field visit

The **Lecture slides** introduce the scenario and its schematic map. In this notebook, you will encode that same fictional pilot visit in Python.

- **Point:** the team's meeting point and an observation stop.
- **LineString:** the **provided illustrative route** connecting them (not a calculated shortest route).
- **Polygon:** a small survey zone around the observation stop, not around the meeting point.

You will create and inspect these geometries, attach attributes, transform coordinates, measure lengths and area, and plot the result. **All data are synthetic, not actual Vienna streets or survey sites.**

# 1. Import the packages

We use **Shapely** for geometry objects, **pandas** for ordinary tables, **GeoPandas** for vector features and CRS handling, and **Matplotlib** for plots.


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon

print("GeoPandas version:", gpd.__version__)


# 2. Python geometry objects — attributes and methods

Create two **Point** instances. An attribute such as `p.x` describes an object; a method such as `p.distance(q)` performs an operation.

These are **arbitrary classroom coordinates**, not real geographic coordinates; their distance is in coordinate units, **not metres**.


In [ ]:
# Tiny examples in an arbitrary, unitless classroom coordinate system.
p = Point(2, 3)
q = Point(5, 7)

print("Class / type:", type(p).__name__)
print("Attributes: x =", p.x, "; y =", p.y, "; geom_type =", p.geom_type)
print("One method: p.distance(q) =", p.distance(q))  # 5 coordinate units, NOT 5 metres


# 3. Three vector geometries — Point, LineString, Polygon

Use synthetic longitude/latitude coordinates near Vienna to encode **the pilot plan introduced in the Lecture**:

- Two **Points**: the meeting place and observation stop.
- One **LineString**: the supplied walk with several turns.
- One **Polygon**: the survey zone around the observation stop. The start is deliberately **outside** this zone.

The source CRS is known to be **EPSG:4326**. Inspect each object's `geom_type`, `is_valid` and `bounds`; later we will calculate useful measurements in a projected CRS.


In [ ]:
# Pilot visit: coordinates are fictional (longitude, latitude), EPSG:4326.
meeting_point = Point(16.3600, 48.2050)
survey_stop = Point(16.3650, 48.2100)

practice_route = LineString([
    (16.3600, 48.2050),
    (16.3620, 48.2050),
    (16.3640, 48.2080),
    (16.3650, 48.2100),
])

# A smaller, irregular pilot survey zone around the stop (not around the start).
practice_area = Polygon([
    (16.3635, 48.2089),
    (16.3662, 48.2086),
    (16.3670, 48.2102),
    (16.3660, 48.2115),
    (16.3637, 48.2112),
])

for name, geom in [
    ("meeting point", meeting_point),
    ("survey stop", survey_stop),
    ("given pilot route", practice_route),
    ("pilot survey zone", practice_area),
]:
    print(name, "| type:", geom.geom_type,
          "| valid:", geom.is_valid,
          "| bounds:", geom.bounds)

print("Survey stop inside pilot zone?", practice_area.covers(survey_stop))
print("Meeting point inside pilot zone?", practice_area.covers(meeting_point))


# 4. Geometry and attributes — GeoDataFrames

A team also needs to know **what each geometry represents**, not just its coordinates. In QGIS/ArcGIS, the attribute table links one row to each feature. We do the same in Python:

- `practice_sites`: two observations/meeting locations (Point layer);
- `practice_routes`: the **provided** pilot walking route (LineString layer);
- `practice_areas`: the planned pilot survey zone (Polygon layer).

First inspect the ordinary pandas attribute table, then build GeoDataFrames with their **known source CRS**. This is the same feature/attribute/geometry relationship seen in the lecture.


In [ ]:
site_attributes = pd.DataFrame({
    "name": ["Meeting point", "Survey stop"],
    "role": ["start", "observation"],
})
print("Ordinary attribute table:")
print(site_attributes)

practice_sites = gpd.GeoDataFrame(
    site_attributes,
    geometry=[meeting_point, survey_stop],
    crs="EPSG:4326",
)
practice_routes = gpd.GeoDataFrame(
    {"name": ["Provided pilot route"]},
    geometry=[practice_route],
    crs="EPSG:4326",
)
practice_areas = gpd.GeoDataFrame(
    {"name": ["Planned pilot survey zone"]},
    geometry=[practice_area],
    crs="EPSG:4326",
)

print("Active geometry column:", practice_sites.geometry.name)
print("Known source CRS:", practice_sites.crs)
practice_sites.head()


# 5. Coordinate reference systems — assign vs transform

For our fictional plan, the source coordinate pairs are **(longitude, latitude)** in **EPSG:4326**. These are degrees; a planar Shapely distance between them is **not a walking distance in metres**.

- **`set_crs()`** records the correct *existing* reference for known coordinates. It **does not** change the coordinate numbers.
- **`to_crs()`** transforms numbers from a known CRS into a new CRS, without changing geographic locations.

For local analysis around Vienna, **EPSG:32633 (WGS 84 / UTM zone 33N)** is a suitable example of a projected CRS in metres. Do not guess or overwrite a source CRS merely because a map appears in the wrong place.


In [ ]:
# These coordinates are known to be longitude/latitude, but the new table
# intentionally has no CRS metadata so we can demonstrate assignment.
missing_crs = gpd.GeoDataFrame(
    {"name": ["Meeting point", "Survey stop"]},
    geometry=[meeting_point, survey_stop],
)
print("Before set_crs:", missing_crs.crs)

labelled = missing_crs.set_crs("EPSG:4326")
print("After set_crs:", labelled.crs)
print("Coordinates unchanged:", labelled.geometry.iloc[0].equals(missing_crs.geometry.iloc[0]))

projected = labelled.to_crs("EPSG:32633")
print("After to_crs:", projected.crs)

coordinate_comparison = pd.DataFrame({
    "name": labelled["name"],
    "longitude_degrees": labelled.geometry.x,
    "latitude_degrees": labelled.geometry.y,
    "easting_m": projected.geometry.x.round(1),
    "northing_m": projected.geometry.y.round(1),
})
coordinate_comparison


# 6. Measurements — why CRS and units matter

The field team needs to know both **how far they would walk along the supplied route** and **how large the planned survey zone is**. These are two different measurements:

- Point-to-point distance: straight-line separation between meeting and survey stops.
- LineString length: length of the **given multi-vertex route** (not a path found by a routing algorithm).
- Polygon area: size of the proposed survey zone.

The GeoPandas/Shapely operations are **planar**. In EPSG:4326, direct distance is in degrees. First transform all three layers to an appropriate local projected CRS, then interpret distances in **metres** and areas in **m²**; projection distortion is not automatically zero.


In [ ]:
distance_in_degrees = practice_sites.geometry.iloc[0].distance(practice_sites.geometry.iloc[1])

sites_utm = practice_sites.to_crs("EPSG:32633")
routes_utm = practice_routes.to_crs("EPSG:32633")
areas_utm = practice_areas.to_crs("EPSG:32633")

straight_distance_m = sites_utm.geometry.iloc[0].distance(sites_utm.geometry.iloc[1])
route_length_m = routes_utm.geometry.length.iloc[0]
study_area_m2 = areas_utm.geometry.area.iloc[0]

print("Direct distance in EPSG:4326:", round(distance_in_degrees, 5), "degrees")
print("Straight-line distance:", round(straight_distance_m), "m")
print("Given route length:", round(route_length_m), "m")
print("Study area:", round(study_area_m2), "m²")
print("Route >= straight line?", route_length_m >= straight_distance_m)


# 7. Plot the pilot survey plan

Instead of drawing the survey polygon as a huge blue background, we want a map that communicates **the actual fieldwork plan**:

1. **Dark point:** where the team meets (outside the survey zone).
2. **Orange route:** the supplied path that the team follows.
3. **Red point:** where observations are taken (inside the survey zone).
4. **Green polygon:** the planned survey zone.

All plotted layers use **EPSG:32633**, so the distances and the small scale bar are in metres. The code includes styling for readability; the three `.plot()` calls and the common CRS are the main ideas.

**Important:** this is a coordinate-based schematic, **not an OpenStreetMap basemap**. It cannot verify that the path follows a walkable street.


In [ ]:
# Plot the three layers in a common projected CRS (EPSG:32633).
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(10, 6))
fig.subplots_adjust(left=0.05, right=0.70, top=0.84, bottom=0.14)
areas_utm.plot(ax=ax, color="#D8F1E6", edgecolor="#167D6B", linewidth=2.2, zorder=1)
routes_utm.plot(ax=ax, color="#DB8542", linewidth=3.4, zorder=2)
sites_utm.plot(ax=ax, color=["#294563", "#D64B43"], markersize=125,
               edgecolor="white", linewidth=1.5, zorder=3)

# Direct labels make it clear which point is which.
start, stop = sites_utm.geometry.iloc[0], sites_utm.geometry.iloc[1]
ax.annotate("Meeting point", (start.x, start.y),
            xytext=(-6, 14), textcoords="offset points", fontsize=10,
            color="#294563", fontweight="bold", ha="right")
ax.annotate("Survey stop", (stop.x, stop.y),
            xytext=(11, -7), textcoords="offset points", fontsize=10,
            color="#A63A35", fontweight="bold")

# Legend and a 200 m scale bar. These lines only improve presentation.
legend_items = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#294563",
           markeredgecolor="white", markersize=10, label="Meeting point"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#D64B43",
           markeredgecolor="white", markersize=10, label="Survey stop"),
    Line2D([0], [0], color="#DB8542", linewidth=3, label="Provided route"),
    Patch(facecolor="#D8F1E6", edgecolor="#167D6B", label="Planned survey zone"),
]
ax.legend(handles=legend_items, loc="upper left", bbox_to_anchor=(1.04, 0.95),
          frameon=False, fontsize=10, title="Map key", title_fontsize=11)

# Keep the scale visible without giant UTM coordinate numbers on the axes.
minx, miny, maxx, maxy = sites_utm.total_bounds
pminx, pminy, pmaxx, pmaxy = areas_utm.total_bounds
minx, miny = min(minx, pminx), min(miny, pminy)
maxx, maxy = max(maxx, pmaxx), max(maxy, pmaxy)
ax.set_xlim(minx - 170, maxx + 250)
ax.set_ylim(miny - 180, maxy + 180)
bar_x, bar_y = minx - 90, miny - 110
ax.plot([bar_x, bar_x + 200], [bar_y, bar_y], color="#334A60", lw=3)
ax.text(bar_x + 100, bar_y - 36, "200 m", ha="center", fontsize=9, color="#334A60")
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title("Pilot field visit  |  A meeting point, route, and survey zone",
             loc="left", fontsize=13, fontweight="bold", pad=15)
fig.text(0.05, 0.055, "Synthetic locations • UTM 33N (metres) • Not a verified street map",
         color="#63748A", fontsize=9)
plt.show()


# 8. Lab checkpoint — Which survey zone should we choose?

**New situation, new data:** after the pilot visit, the same university team is considering **three alternative zones for a real-world-style (but fictional) urban microclimate field survey**.

For each option A, B and C, you are given:

- a **candidate observation stop** (Point);
- a **predefined possible walking route** from the common meeting point (LineString);
- a **candidate survey zone** (Polygon): a simplified planning boundary, not an observed park or city block.

**Planning rule:** the zone must be at least **18,000 m²** to allow enough space for observations. **Among qualifying zones, choose the option with the shortest *given route*.** The aim is to **justify a decision** with geographic data, not to find a new route.

**Do this independently, using the techniques from Sections 1–7:** build the three geometry layers and their attributes, use the known source CRS (EPSG:4326), choose and apply a suitable local projected CRS, calculate each zone's area and route length, make a clearly labelled map and a short comparison table, and explain your recommendation with units. Your notebook should not rely on external downloads or new topics such as spatial joins, buffering, or network routing.


## Checkpoint — a new planning decision

The **Lecture slides** introduce three fictional alternatives, **A, B and C**. Each option has an observation stop, a **given route** from the same meeting point, and a proposed survey zone.

**Decision rule:** first check which zones are at least **18,000 m²**; then recommend the qualifying option with the **shortest given route**. You will calculate the results from the coordinates below. No real street network or routing algorithm is involved.

## Checkpoint data — three alternative field plans

The supplied coordinate pairs use **(longitude, latitude) in EPSG:4326** and are synthetic teaching examples. The shared `meeting_xy` is the team's start. Each candidate ID in `candidate_xy`, `given_route_xy`, and `survey_area_xy` refers to the **same survey option**.

The polygons are simplified candidate zones, not actual green spaces. The LineStrings are **given paths**, not road-network routes. **Keep these input coordinates unchanged** so results can be compared consistently.


In [ ]:
# Known geographic coordinates, EPSG:4326, around Vienna.
meeting_xy = (16.3540, 48.2070)

candidate_xy = {
    "A": (16.3610, 48.2110),
    "B": (16.3660, 48.2100),
    "C": (16.3650, 48.1980),
}

given_route_xy = {
    "A": [meeting_xy, (16.3540, 48.2100), (16.3580, 48.2100), candidate_xy["A"]],
    "B": [meeting_xy, (16.3570, 48.2070), (16.3590, 48.2090),
          (16.3630, 48.2090), candidate_xy["B"]],
    "C": [meeting_xy, (16.3550, 48.2040), (16.3560, 48.2010),
          (16.3610, 48.2010), candidate_xy["C"]],
}

# Three small rectangles; each has a different survey area.
survey_area_xy = {
    "A": [(16.3602, 48.2105), (16.3618, 48.2105),
          (16.3618, 48.2115), (16.3602, 48.2115)],
    "B": [(16.36485, 48.2094), (16.36715, 48.2094),
          (16.36715, 48.2106), (16.36485, 48.2106)],
    "C": [(16.3636, 48.19735), (16.3664, 48.19735),
          (16.3664, 48.19865), (16.3636, 48.19865)],
}

area_requirement_m2 = 18_000
print("Candidate IDs:", list(candidate_xy))
print("Minimum qualifying area:", area_requirement_m2, "m²")


## Complete the checkpoint analysis

Write your own analysis in the next code cell; add extra cells if helpful. Adapt the geometry → GeoDataFrame → `to_crs()` → measurement → plot pattern from the pilot visit, but apply it to the **new A/B/C inputs**.

To start, here is one possible way to build the **site** layer:

```python
sites = gpd.GeoDataFrame(
    {"site": ["A", "B", "C"]},
    geometry=[Point(candidate_xy["A"]), Point(candidate_xy["B"]), Point(candidate_xy["C"])],
    crs="EPSG:4326",
)
```

Next, build the **given-route** and **candidate-zone** layers using the matching ID keys. From those, calculate route length and area in the **same appropriate projected CRS**. Your result should include a table with columns for ID, area (m²), route length (m) and whether the site qualifies.

For your plot, **use the pilot map as a style reference**, not just its data: distinguish points, lines and polygons with meaningful colors, include labels and a legend, and remove confusing offset notation. A good map should help explain **which** option you selected, not just prove that `.plot()` works.

**Optional challenge:** compare the straight-line distance to one stop with the length of its supplied route. Why are the two lengths different?


In [ ]:
# CHECKPOINT — write your own code BELOW (insert one or more cells).
# 1) Geometry objects + GeoDataFrames for A / B / C
# 2) CRS transformation
# 3) Route length + area comparison (including whether each meets 18,000 m²)
# 4) One labelled plot


## Complete the checkpoint notes

Double-click this Markdown cell and replace the placeholders below.

**1. Which site do you recommend, and which sites qualify?**  
YOUR ANSWER

**2. What numerical evidence supports your choice?**  
Survey area(s) = ... m² ; Given route length(s) = ... m.

**3. Which CRS did you use for the measurements, and why?**  
YOUR ANSWER

**4. Why is directly reading distances in EPSG:4326 as metres incorrect?**  
YOUR ANSWER


## Checkpoint requirements

- [ ] The guided pilot visit runs and the map clearly distinguishes **meeting point, route, observation stop and survey zone**.
- [ ] The checkpoint includes the **three separate feature layers**: candidate stops (Point), given routes (LineString) and survey zones (Polygon), with matching candidate IDs.
- [ ] You calculated **survey area (m²)** and **given route length (m)** using a suitable **projected CRS**.
- [ ] Your comparison table shows which options pass the **18,000 m²** threshold, and you recommend the shortest **given route** among them.
- [ ] Your **labelled map** uses different symbols/colors and a legend. The plot is geographically consistent with the CRS.
- [ ] The four checkpoint notes explain your recommendation, numerical evidence and CRS reasoning.

Keep the outputs visible in the saved notebook.


## Submit your notebook to Moodle

1. Save the notebook **after** your final run.
2. Check that the feature layers, comparison table, map, and checkpoint notes are visible.
3. Submit the completed `.ipynb` file to the **Week 2 lab checkpoint** on Moodle.
